# 02 — Stability audit (provisional)

**Status: `provisional_holdout_not_integrated`.** The audit uses the inner 2016–17 period plus burned retrospective views (2018–22 and COVID 2020–21). It was frozen at **2026-10-23T06:31:40Z**, before the one-shot holdout completed at **2026-10-23T07:45:19Z**. The holdout completed later and was **not integrated** (`available_at_audit: false`, `completed_later: true`, `used: false`); the audit was **not re-run on the holdout**. Every number here is descriptive/provisional and must not be written as a confirmatory headline. The shipped stability files are the holdout-excluded 50-refit rerun; the holdout was not used.

**What is audited.**

- Exact additive decomposition of the 49-feature logistic decision function.
- Source- and target-actor views, including their redundancy.
- Fifty dyad-cluster bootstrap refits ("seed" replicates with C fixed), not optimizer seeds.
- Period/regime and D2 window contrasts.
- Top-k churn and a decision proxy.
- HGB block occlusion as a model-class check.

**Not done.** No fitting, no new data, no ICEWS event-panel reads, no row-level predictions.


In [1]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "results" / "framework_pilot").is_dir():
            return cand
    return here


ROOT = find_repo_root()
STABILITY = ROOT / "results" / "framework_pilot" / "stability"
TABLES = STABILITY / "tables"


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def load_json(path: Path) -> dict:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return {}
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return pd.DataFrame()
    return pd.read_csv(path)


print("repo root:", rel(ROOT))
print("stability artifacts:", "present" if STABILITY.is_dir() else "MISSING")


repo root: .
stability artifacts: present


## Audit scope and provenance

The periods are displayed exactly as recorded in `summary.json`. "Burned retrospective" means the period had already been observed by earlier exploratory runs; it is not a fresh test.

The holdout is a separate, later artifact. This audit was frozen before that holdout completed and does not use it.


In [2]:
summary = load_json(STABILITY / "summary.json")

if summary:
    print("status:", summary.get("status"))
    holdout = summary.get("holdout", {})
    print("holdout available at audit:", holdout.get("available_at_audit"))
    print("holdout completed later:", holdout.get("completed_later"))
    print("holdout used:", holdout.get("used"))
    periods = pd.DataFrame(summary.get("scope", {}).get("periods", []))
    if not periods.empty:
        display(periods)
    exact = summary.get("exactness", {})
    print("exactness verdict:", exact.get("verdict"))
    print(
        "decision-function max abs error:",
        exact.get("decision_function_max_abs_error"),
    )
    print("source/target note:", summary.get("source_target", {}).get("interpretation"))
else:
    print("No stability summary available.")


status: provisional_holdout_not_integrated
holdout available at audit: False
holdout completed later: True
holdout used: False


,name,lo,hi,label,status
0,inner_2016_17,252,274,"2016-01..2017-11 (inner validation, embargoed)",inner_validation
1,retro_2018_22,276,334,"2018-01..2022-11 (burned retrospective, second...",burned_retrospective
2,covid_2020_21,300,323,"2020-01..2021-12 (COVID sub-window, secondary)",burned_retrospective


exactness verdict: PASS
decision-function max abs error: 1.2886362013020405e-06
source/target note: Attribution is relative to the training mean; positive values push the dyad above average and do not imply high absolute risk.


## Exactness

The identity is **intercept + 49 feature contributions = decision function**. The maximum absolute error is 1.29×10⁻⁶ in log-odds. The larger raw-vs-logit error (3.12×10⁻⁵) is a float32 `predict_proba` round-trip, not a decomposition failure. Temperature is T=1.0, so the temperature track coincides with the raw track; isotonic is excluded because it is not affine.


In [3]:
exactness = load_csv(TABLES / "exactness_check.csv")

if not exactness.empty:
    etab = exactness[
        [
            "period", "n_rows", "decision_max_abs_error",
            "temperature_max_abs_error", "accumulator_max_abs_error",
            "accumulator_rows_above_tol",
        ]
    ].rename(
        columns={
            "period": "Period",
            "n_rows": "Rows",
            "decision_max_abs_error": "Decision max abs error",
            "temperature_max_abs_error": "Temperature max abs error",
            "accumulator_max_abs_error": "Stream max abs error",
            "accumulator_rows_above_tol": "Stream rows > 1e-6",
        }
    )
    display(etab)
else:
    print("Exactness table not available.")


,Period,Rows,Decision max abs error,Temperature max abs error,Stream max abs error,Stream rows > 1e-6
0,inner_2016_17,50000,0.000001,3.819167e-14,0.000029,3029
1,retro_2018_22,50000,0.000001,6.306067e-14,0.000201,6886
2,covid_2020_21,50000,0.000001,3.019807e-14,0.000038,2226


## Actor buckets and source/target redundancy

The bucket shares are shares of the mean absolute contribution, not variance shares and not risk. Source and target actor views carry about 45% of the absolute contribution together. At the actor level they correlate at r ≈ 0.998, so they are **not independent evidence**; report both, but do not double-count them.

Positive attribution means "less negative than the training mean", not "high absolute risk". The top positive actors are low-activity microstates with mean predicted probabilities far below the period base rate.


In [4]:
buckets = load_csv(TABLES / "bucket_shares.csv")

if not buckets.empty:
    btab = buckets[buckets["view"] == "all_features"].pivot(
        index="bucket", columns="period", values="abs_share"
    )
    period_order = ["inner_2016_17", "retro_2018_22", "covid_2020_21"]
    btab = btab.reindex(
        columns=[p for p in period_order if p in btab.columns]
    )
    btab = btab.round(4)
    display(btab)
else:
    print("Bucket-share table not available.")

corr = pd.DataFrame(
    summary.get("source_target", {}).get("source_target_correlation", [])
)
if not corr.empty:
    ctab = corr[["period", "n", "pearson", "spearman", "mean_abs_diff"]].copy()
    for col in ["pearson", "spearman", "mean_abs_diff"]:
        ctab[col] = ctab[col].astype(float).round(4)
    display(ctab)
else:
    print("Source/target correlation table not available.")


period,inner_2016_17,retro_2018_22,covid_2020_21
bucket,,,
calendar,0.0280,0.0344,0.0356
dyad,0.2974,0.2929,0.2876
graph,0.1696,0.1689,0.1700
source_actor,0.2158,0.2154,0.2166
target_actor,0.2335,0.2333,0.2349
train_stats,0.0557,0.0551,0.0553


,period,n,pearson,spearman,mean_abs_diff
0,inner_2016_17,196,0.9979,0.9976,0.0364
1,retro_2018_22,196,0.9985,0.9984,0.0386
2,covid_2020_21,196,0.9984,0.9981,0.0435


## Variance decomposition: regime, not seed

These are within-actor variance shares. Seed noise is 5.3% (target) to 9.9% (source); period/regime is 32.3–33.5%; the actor×time residual is 57.7–61.2%. The median regime/seed variance ratio is 3.22 (source) and 5.34 (target); 61.7–70.4% of actors have a ratio above 2. This is the central stability finding: the audit is **regime-limited**, not seed-limited.


In [5]:
variance = load_csv(TABLES / "variance_decomposition.csv")

if not variance.empty:
    vtab = variance[variance["level"] == "within_actor_total"].pivot(
        index="component", columns="role", values="share"
    )
    vtab = vtab.round(4)
    display(vtab)

    ratio = variance[
        (variance["level"] == "actor_ratio")
        & (variance["component"].isin(
            ["regime_var_over_seed_var_median", "share_actors_ratio_gt_2"]
        ))
    ]
    if not ratio.empty:
        rtab = ratio.pivot(
            index="component", columns="role", values="share"
        )
        display(rtab.round(3))
else:
    print("Variance-decomposition table not available.")


role,source,target
component,,
actor_x_time_resid,0.5773,0.6125
period_regime,0.3232,0.3346
seed_noise,0.0994,0.0529


role,source,target
component,,
regime_var_over_seed_var_median,3.224,5.344
share_actors_ratio_gt_2,0.617,0.704


## Top-k churn: cluster bootstrap, period, window

The 50 "seeds" are dyad-cluster bootstrap refits with C fixed, not optimizer seeds. Signed top-k Jaccard is the decision-relevant measure. At k=10, the mean seed-pair Jaccard is 0.88–0.91 for attribution and 0.99–1.00 for risk (worst pairs 0.66 and 0.98); period pairs fall to 0.54–0.67; D2 window pairs fall as low as 0.25. Rank correlation stays high (seed-pair Spearman ≥0.987; period 0.95–0.99), but the **exact membership** of a 10-actor review set is not stable across regimes or windows.

Cold-start/onset explanations should not be pooled with stable-history cases.


In [6]:
churn = load_csv(TABLES / "topk_churn_actor_month.csv")

if not churn.empty:
    ctab = churn[churn["k"] == 10][
        [
            "period", "role", "score", "seed_pair_mean_jaccard",
            "seed_pair_min", "mean_churn",
            "median_boundary_gap_over_seed_sd",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "score": "Score",
            "seed_pair_mean_jaccard": "Seed Jaccard@10",
            "seed_pair_min": "Seed Jaccard@10 min",
            "mean_churn": "Mean churn",
            "median_boundary_gap_over_seed_sd": "Boundary gap / seed SD",
        }
    )
    for col in ["Seed Jaccard@10", "Seed Jaccard@10 min", "Mean churn"]:
        ctab[col] = ctab[col].astype(float).round(3)
    ctab = ctab.reset_index(drop=True)
    display(ctab)
else:
    print("Top-k churn table not available.")

periods = load_csv(TABLES / "stability_periods.csv")
if not periods.empty:
    ptab = periods[
        ["role", "period_a", "period_b", "spearman",
         "jaccard_pos@10", "sign_agreement"]
    ].rename(
        columns={
            "role": "View",
            "period_a": "Period A",
            "period_b": "Period B",
            "spearman": "Spearman",
            "jaccard_pos@10": "Signed Jaccard@10",
            "sign_agreement": "Sign agreement",
        }
    )
    for col in ["Spearman", "Signed Jaccard@10", "Sign agreement"]:
        ptab[col] = ptab[col].astype(float).round(3)
    ptab = ptab.reset_index(drop=True)
    display(ptab)
else:
    print("Period-stability table not available.")


,Period,View,Score,Seed Jaccard@10,Seed Jaccard@10 min,Mean churn,Boundary gap / seed SD
0,inner_2016_17,source,attribution,0.876,0.662,0.124,0.392
1,inner_2016_17,source,risk,0.993,0.976,0.007,13.631
2,inner_2016_17,target,attribution,0.901,0.733,0.099,0.771
3,inner_2016_17,target,risk,0.999,0.992,0.001,11.005
4,retro_2018_22,source,attribution,0.889,0.705,0.111,0.582
5,retro_2018_22,source,risk,0.991,0.978,0.009,10.568
6,retro_2018_22,target,attribution,0.904,0.725,0.096,0.676
7,retro_2018_22,target,risk,0.996,0.991,0.004,18.012
8,covid_2020_21,source,attribution,0.906,0.740,0.094,0.682
9,covid_2020_21,source,risk,0.996,0.985,0.004,9.754


,View,Period A,Period B,Spearman,Signed Jaccard@10,Sign agreement
0,source,inner_2016_17,retro_2018_22,0.974,0.667,0.959
1,source,inner_2016_17,covid_2020_21,0.965,0.538,0.944
2,source,retro_2018_22,covid_2020_21,0.992,0.667,0.964
3,target,inner_2016_17,retro_2018_22,0.964,0.667,0.939
4,target,inner_2016_17,covid_2020_21,0.952,0.538,0.929
5,target,retro_2018_22,covid_2020_21,0.989,0.667,0.969


In [7]:
windows = load_csv(TABLES / "stability_windows.csv")

if not windows.empty:
    wtab = (
        windows.groupby(["period", "role", "window_a", "window_b"])
        .agg(
            spearman=("spearman", "median"),
            jaccard_pos10=("jaccard_pos@10", "median"),
            sign_agreement=("sign_agreement", "median"),
        )
        .reset_index()
        .rename(
            columns={
                "period": "Period",
                "role": "View",
                "window_a": "Window A",
                "window_b": "Window B",
                "spearman": "Spearman (median)",
                "jaccard_pos10": "Signed Jaccard@10 (median)",
                "sign_agreement": "Sign agreement (median)",
            }
        )
    )
    for col in [
        "Spearman (median)", "Signed Jaccard@10 (median)",
        "Sign agreement (median)",
    ]:
        wtab[col] = wtab[col].astype(float).round(3)
    wtab = wtab.reset_index(drop=True)
    display(wtab)
else:
    print("Window-stability table not available.")


,Period,View,Window A,Window B,Spearman (median),Signed Jaccard@10 (median),Sign agreement (median)
0,inner_2016_17,source,W_b_dyad_onset,W_c_dormancy_reentry,0.927,0.538,0.852
1,inner_2016_17,source,W_b_dyad_onset,W_d_stable,0.924,0.538,0.929
2,inner_2016_17,source,W_b_dyad_onset,full_period,0.937,0.538,0.942
3,inner_2016_17,source,W_c_dormancy_reentry,W_d_stable,0.981,0.667,0.903
4,inner_2016_17,source,W_c_dormancy_reentry,full_period,0.988,0.667,0.908
5,inner_2016_17,source,W_d_stable,full_period,0.998,1.000,0.985
6,inner_2016_17,target,W_b_dyad_onset,W_c_dormancy_reentry,0.901,0.250,0.836
7,inner_2016_17,target,W_b_dyad_onset,W_d_stable,0.921,0.538,0.920
8,inner_2016_17,target,W_b_dyad_onset,full_period,0.931,0.538,0.931
9,inner_2016_17,target,W_c_dormancy_reentry,W_d_stable,0.964,0.429,0.887


## Decision proxy: risk selects, attribution explains

The risk alert ranks actor-months by mean predicted probability. The attribution alert ranks them by the actor's own bucket contribution. These are different objects. Across the reported k values, risk lift is 3.20–3.44 for actor-month review and 7.44–9.05 for dyad review, with near-unit seed Jaccard. Attribution lift is 0.14–0.16 (actor-month) and 0.04–0.56 (dyad), with essentially zero overlap with the risk alert.

The attribution top-k is therefore **anti-useful as an alert rule**. That is a negative decision-impact result, not a failed stability result. Use risk to choose what to review, then attach the exact decomposition to explain why.


In [8]:
decision = load_csv(TABLES / "decision_impact_actor_month.csv")

if not decision.empty:
    dtab = decision[decision["k"] == 10][
        [
            "period", "role", "alert", "mean_lift_vs_month_base",
            "lift_ci_lo", "lift_ci_hi",
            "mean_jaccard_with_risk_alert",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "alert": "Alert",
            "mean_lift_vs_month_base": "Mean lift",
            "lift_ci_lo": "Lift CI low",
            "lift_ci_hi": "Lift CI high",
            "mean_jaccard_with_risk_alert": "Jaccard with risk",
        }
    )
    for col in ["Mean lift", "Lift CI low", "Lift CI high", "Jaccard with risk"]:
        dtab[col] = dtab[col].astype(float).round(3)
    dtab = dtab.reset_index(drop=True)
    display(dtab)
else:
    print("Actor-month decision table not available.")

dyad = load_csv(TABLES / "decision_impact_dyad.csv")
if not dyad.empty:
    ytab = dyad[dyad["k"].isin([10, 100, 1000])][
        [
            "alert", "k", "precision", "recall", "lift_vs_base",
            "jaccard_with_risk",
        ]
    ].rename(
        columns={
            "alert": "Alert",
            "k": "k",
            "precision": "Precision",
            "recall": "Recall",
            "lift_vs_base": "Lift",
            "jaccard_with_risk": "Jaccard with risk",
        }
    )
    for col in ["Precision", "Recall", "Lift", "Jaccard with risk"]:
        ytab[col] = ytab[col].astype(float).round(3)
    ytab = ytab.reset_index(drop=True)
    display(ytab)
else:
    print("Dyad decision table not available.")


,Period,View,Alert,Mean lift,Lift CI low,Lift CI high,Jaccard with risk
0,inner_2016_17,source,attribution,0.139,0.117,0.164,0.0
1,inner_2016_17,source,risk,3.220,3.175,3.257,1.0
2,inner_2016_17,target,attribution,0.151,0.116,0.184,0.0
3,inner_2016_17,target,risk,3.203,3.163,3.253,1.0
4,retro_2018_22,source,attribution,0.152,0.134,0.178,0.0
5,retro_2018_22,source,risk,3.327,3.284,3.361,1.0
6,retro_2018_22,target,attribution,0.160,0.141,0.181,0.0
7,retro_2018_22,target,risk,3.256,3.217,3.296,1.0
8,covid_2020_21,source,attribution,0.136,0.109,0.169,0.0
9,covid_2020_21,source,risk,3.437,3.359,3.524,1.0


,Alert,k,Precision,Recall,Lift,Jaccard with risk
0,risk,10,1.000,0.004,9.053,1.000
1,risk,100,1.000,0.043,9.049,1.000
2,risk,1000,0.822,0.352,7.439,1.000
3,total,10,1.000,0.004,9.053,0.667
4,total,100,1.000,0.043,9.049,0.961
5,total,1000,0.822,0.352,7.443,0.994
6,source,10,0.013,0.000,0.118,0.000
7,source,100,0.021,0.001,0.189,0.000
8,source,1000,0.055,0.023,0.495,0.006
9,target,10,0.004,0.000,0.039,0.000


## HGB block occlusion

HGB is a separate refit (150 iterations, current graph cache), not the 120-iteration holdout capacity control. Occlusion–occlusion Spearman is 0.51–0.67 for the source view and 0.85–0.87 for the target view: broad rank agreement, stronger for targets. At actor level, top-10 absolute overlap is at most 0.05 for source and 0.18–0.33 for target, so model-class disagreement is concentrated in the source tail. This is descriptive agreement, not an HGB-vs-logistic superiority or parity claim.


In [9]:
hgb = load_csv(TABLES / "hgb_occlusion_agreement.csv")

if not hgb.empty:
    htab = hgb[hgb["level"] == "actor"][
        [
            "period", "role", "spearman_logocc_hgb",
            "spearman_logocc_ci_lo", "spearman_logocc_ci_hi",
            "sign_logocc_hgb", "jaccard_abs@10",
            "spearman_abs_attr_hgb",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "spearman_logocc_hgb": "Occlusion-occlusion Spearman",
            "spearman_logocc_ci_lo": "CI low",
            "spearman_logocc_ci_hi": "CI high",
            "sign_logocc_hgb": "Sign agreement",
            "jaccard_abs@10": "Top-10 abs occlusion Jaccard",
            "spearman_abs_attr_hgb": "Abs attribution vs HGB",
        }
    )
    for col in [
        "Occlusion-occlusion Spearman", "CI low", "CI high",
        "Sign agreement", "Top-10 abs occlusion Jaccard",
        "Abs attribution vs HGB",
    ]:
        htab[col] = htab[col].astype(float).round(3)
    htab = htab.reset_index(drop=True)
    display(htab)
else:
    print("HGB occlusion table not available.")


,Period,View,Occlusion-occlusion Spearman,CI low,CI high,Sign agreement,Top-10 abs occlusion Jaccard,Abs attribution vs HGB
0,inner_2016_17,source,0.542,0.421,0.644,0.750,0.053,0.319
1,inner_2016_17,target,0.862,0.802,0.907,0.852,0.333,0.776
2,retro_2018_22,source,0.511,0.361,0.627,0.730,0.000,0.406
3,retro_2018_22,target,0.847,0.776,0.898,0.801,0.176,0.807
4,covid_2020_21,source,0.665,0.543,0.759,0.735,0.053,0.471
5,covid_2020_21,target,0.868,0.803,0.908,0.791,0.250,0.811


## Provisional bottom line

- Exactness **passes**: the additive decomposition reproduces the decision function to 1.29×10⁻⁶.
- Actor views are **stable to cluster resampling** but **regime-limited**: seed noise is 5.3–9.9% of within-actor variance, regime is 32.3–33.5%.
- Source and target views are **not independent** (r ≈ 0.998).
- Attribution top-k is **not an alert rule**; risk is. Post-hoc explanation inside a risk alert is stable (Spearman ≥0.997).
- The holdout **completed after this audit was frozen** and was **not integrated** (`available_at_audit: false`, `completed_later: true`, `used: false`). The audit was **not re-run** on it, and none of these numbers is a confirmatory holdout result.
